# Enfoque con Embeddings
Todo a embeddings (queries, sinopsis + año + director + keywords) --> 5 con mas similtud coseno (comparando queries vs sinopsis + año + director + keywords)

1. unificar texto
2. embeddings con w2v o sentence transformer sobre texto y queries
3. similitud coseno text vs queries

In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.1 MB/s eta 0:00:00:00:0100:01


In [24]:
from datasets import load_dataset
import pandas as pd
import re       # libreria de expresiones regulares
import string   # libreria de cadena de caracteres
from gensim.models.phrases import Phrases, Phraser
import multiprocessing
from gensim.models import Word2Vec
import numpy as np

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [4]:
sinopsis = load_dataset("mathigatti/spanish_imdb_synopsis")

Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [6]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/usuarios/usuarios.csv")

In [11]:
usuarios.head()

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...
2,U03,Camila,definido,Los padres de él,Mamá a la fuerza,Norbit,Elizabethtown,My Sassy Girl,Una comedia donde la relación entre dos person...
3,U04,Tomás,definido,Matrix,Fahrenheit 451,¡Olvídate de mí!,X-Men,El único,Algo que haga pensar sobre qué es real y qué e...
4,U05,Lucía,definido,La novia cadáver,Spirit: El corcel indomable,Las aventuras de Peabody y Sherman,Los Increíbles,Steamboy,Animación donde el protagonista lucha por su l...


Visualizamos las queries

In [10]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [12]:
df_pelis = pd.DataFrame(sinopsis['train'])
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


## Preprocesado

Defino una funcion para limpiar texto

In [16]:
def limpiar_texto(text):
    # pasa las mayusculas del texto a minusculas
    text = text.lower()
    # reemplaza texto entre corchetes por espacio en blanco
    text = re.sub(r'\[.*?¿\]%', ' ', text)
    # reemplaza signos de puntuacion por espacio en blanco
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    # remueve palabras que contienen numeros.
    text = re.sub(r'\w*\d\w*', '', text)
    # remueve caracteres especiales y saltos de linea
    text = re.sub('[‘’“”…«»]', '', text)
    text = re.sub('\n', ' ', text)
    return text

Unificamos las variables relevantes en un texto

In [13]:
df_pelis["texto"] = (
    df_pelis["name"] + " "
    + df_pelis["description"] + " "
    + df_pelis["year"].fillna('').astype(str) + " "
    + df_pelis["director"].fillna('') + " "
    + df_pelis["genre"] + " "
    + df_pelis["keywords"]
)

In [17]:
limpieza = lambda x: limpiar_texto(x)
data_clean = pd.DataFrame(df_pelis["texto"].apply(limpieza))

Agregamos bigramas al corpus (?)

In [20]:
input = [row.split() for row in data_clean["texto"]] # separamos en una lista
phrases = Phrases(input, min_count=20, progress_per=1000)

bigram = Phraser(phrases)

sentences = bigram[input]

## Embedding de peliculas

### Entrenamos modelo World2Vec

> [!!!] falta elegir los parametros del modelo acorde al trabajo.

In [23]:
cores = multiprocessing.cpu_count()

w2v_model = Word2Vec(min_count=20, # ignora palabras cuya frecuencia es menor a esta
                     window=2, # tamanio de la ventana de contexto
                     vector_size=300, # dimension del embedding
                     sample=6e-5, # umbral para downsamplear palabras muy frecuentes
                     alpha=0.03, # tasa de aprendizaje inicial (entrenamiento de la red neuronal)
                     min_alpha=0.0007, # tasa de aprendizaje minima
                     negative=20, # penalidad de palabras muy frecuentes o poco informaitvas
                     workers=cores) # numero de cores para entrenar el modelo

w2v_model.build_vocab(sentences, progress_per=10000) # construye el vocabulario

### ENTRENA EL MODELO
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

(1274469, 6144450)

### Calcular vector promedio de cada película

Definimos una funcion que calcula el vector promedio a partir de un texto

In [25]:
def obtener_vector_promedio(texto, modelo):
    palabras = texto.split()

    vectores_palabras = [modelo.wv[palabra] for palabra in palabras if palabra in modelo.wv]

    if not vectores_palabras:
        return np.zeros(modelo.wv.vector_size)

    return np.mean(vectores_palabras, axis=0)

Calculamos el embedding promedio de cada pelicula

In [31]:
embeddings_peliculas = pd.DataFrame(np.array([obtener_vector_promedio(text, w2v_model) for text in data_clean['texto']]))

In [32]:
embeddings_peliculas.head()

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,0.003052,-0.068935,0.112246,0.171325,-0.096816,0.066071,-0.075511,0.271405,0.031434,0.025390,...,0.041415,0.061093,0.333886,-0.105234,0.173369,0.083610,0.133110,-0.069436,0.132140,0.014020
1,0.024492,-0.058410,0.125399,0.236618,-0.124291,0.064446,-0.090625,0.243767,-0.003063,0.015505,...,0.064443,0.133871,0.278527,-0.113368,0.170888,0.094499,0.143409,-0.058470,0.126272,-0.008772
2,0.006154,-0.058569,0.064328,0.234473,-0.127537,0.043643,-0.094651,0.237292,0.003293,-0.042800,...,0.047043,0.143121,0.259836,-0.094910,0.188148,0.091868,0.091561,-0.067068,0.110938,-0.028709
3,0.023215,-0.034390,0.092157,0.227685,-0.115871,0.047445,-0.095807,0.270160,0.001356,-0.009597,...,0.054682,0.126048,0.273670,-0.105818,0.173109,0.104777,0.118783,-0.072644,0.117439,-0.003041
4,0.022511,-0.082786,0.149495,0.237061,-0.113630,0.065525,-0.112229,0.218235,0.016179,0.028508,...,0.064422,0.130384,0.304234,-0.116733,0.172434,0.082668,0.138963,-0.043569,0.127838,-0.001501


Estan en orden entonces podemos joinear por index

In [33]:
pelis_embd = embeddings_peliculas.merge(df_pelis[["name","id"]], left_index=True, right_index=True)
pelis_embd.head()

,0,1,2,3,4,5,6,7,8,9,...,292,293,294,295,296,297,298,299,name,id
0,0.003052,-0.068935,0.112246,0.171325,-0.096816,0.066071,-0.075511,0.271405,0.031434,0.025390,...,0.333886,-0.105234,0.173369,0.083610,0.133110,-0.069436,0.132140,0.014020,Herida abierta,1
1,0.024492,-0.058410,0.125399,0.236618,-0.124291,0.064446,-0.090625,0.243767,-0.003063,0.015505,...,0.278527,-0.113368,0.170888,0.094499,0.143409,-0.058470,0.126272,-0.008772,"Elvira, reina de las tinieblas",2
2,0.006154,-0.058569,0.064328,0.234473,-0.127537,0.043643,-0.094651,0.237292,0.003293,-0.042800,...,0.259836,-0.094910,0.188148,0.091868,0.091561,-0.067068,0.110938,-0.028709,Durmiendo con su enemigo,3
3,0.023215,-0.034390,0.092157,0.227685,-0.115871,0.047445,-0.095807,0.270160,0.001356,-0.009597,...,0.273670,-0.105818,0.173109,0.104777,0.118783,-0.072644,0.117439,-0.003041,Elizabethtown,4
4,0.022511,-0.082786,0.149495,0.237061,-0.113630,0.065525,-0.112229,0.218235,0.016179,0.028508,...,0.304234,-0.116733,0.172434,0.082668,0.138963,-0.043569,0.127838,-0.001501,Godzilla,5


## Embeddings de usuarios

Limpiamos las queries con el mismo proceso de antes

In [34]:
data_clean_users = pd.DataFrame(usuarios["query"].apply(limpieza))

### Embedding de las queries

In [35]:
embeddings_query = pd.DataFrame()

for query in data_clean_users["query"]:
    words = query.split()
    words_embeddings = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    embedding_mean = np.mean(words_embeddings, axis=0)
    embeddings_query = pd.concat([embeddings_query, pd.DataFrame(embedding_mean).T])

embeddings_query.reset_index(drop=True,inplace=True)

In [36]:
embeddings_query = embeddings_query.merge(usuarios[["id"]], left_index=True, right_index=True)
embeddings_query

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,id
0,-0.004298,-0.073146,0.110788,0.261349,-0.145625,0.050732,-0.080550,0.241608,-0.020588,-0.019952,...,0.149603,0.278452,-0.128087,0.170934,0.087102,0.142092,-0.050823,0.126011,-0.027833,U01
1,0.001660,-0.061452,0.103090,0.222772,-0.107933,0.074199,-0.084731,0.241127,-0.000553,-0.005523,...,0.133072,0.264866,-0.111810,0.176449,0.096136,0.115202,-0.062822,0.118213,-0.010879,U02
2,0.004058,-0.075192,0.114444,0.271899,-0.135906,0.051688,-0.100725,0.265677,-0.023602,-0.010702,...,0.162061,0.267048,-0.126513,0.185012,0.087501,0.149518,-0.058624,0.133195,-0.030699,U03
3,0.020188,-0.102010,0.136850,0.273310,-0.144361,0.073167,-0.097958,0.222833,-0.014371,-0.025233,...,0.196801,0.278708,-0.126519,0.189170,0.087051,0.129287,-0.033607,0.167780,-0.028668,U04
4,0.024172,-0.062652,0.137710,0.243087,-0.128460,0.069601,-0.095405,0.239623,-0.000220,0.005631,...,0.158504,0.279309,-0.120472,0.172868,0.099202,0.134333,-0.050910,0.151732,-0.013663,U05
5,-0.008874,-0.111073,0.132556,0.253786,-0.140566,0.084166,-0.095888,0.241434,-0.007622,-0.016240,...,0.183831,0.290300,-0.123464,0.187833,0.069468,0.152119,-0.046392,0.157030,-0.029182,U06
6,0.009897,-0.081073,0.127455,0.281254,-0.143947,0.059923,-0.102259,0.242886,-0.024189,-0.007803,...,0.180199,0.263471,-0.130582,0.186656,0.092185,0.139311,-0.042659,0.137625,-0.034210,U07
7,0.014921,-0.105028,0.142912,0.231994,-0.128238,0.077990,-0.094373,0.215219,0.014131,-0.004460,...,0.155518,0.306607,-0.120050,0.175372,0.069030,0.126621,-0.038364,0.152305,-0.007900,U08
8,-0.006906,-0.115716,0.139287,0.283795,-0.148305,0.075316,-0.090540,0.241122,-0.024719,-0.021136,...,0.195240,0.279504,-0.138102,0.190409,0.079441,0.155422,-0.039853,0.154316,-0.036408,U09
9,0.012608,-0.092987,0.121881,0.273645,-0.147382,0.053133,-0.090834,0.244204,-0.024346,-0.035127,...,0.186890,0.266308,-0.128627,0.186197,0.088987,0.134007,-0.041129,0.155720,-0.032797,U10


### Embedding historial

In [45]:
def calcular_embedding_historial(usuario, pelis_embd):
    peliculas_usuario = usuario[['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']].tolist()
    embeddings = []
    for pelicula in peliculas_usuario:
        # check pelicula in pelis_embd
        if pelicula not in pelis_embd['name'].values:
            print(f"Película '{pelicula}' no encontrada en el DataFrame de embeddings.")
        embedding = pelis_embd[pelis_embd['name'] == pelicula].iloc[0][:-2].to_numpy(dtype=np.float32)
        embeddings.append(embedding)
    historial_embedding = np.mean(embeddings, axis=0)
    return historial_embedding

In [47]:
historiales_embeddings = []
for index, usuario in usuarios.iterrows():
    historial_embedding = calcular_embedding_historial(usuario, pelis_embd)
    historial_embedding = np.append(historial_embedding, usuario['id'])
    
    historiales_embeddings.append(historial_embedding)

historiales_df = pd.DataFrame(historiales_embeddings)

Película 'Walker Texas Ranger' no encontrada en el DataFrame de embeddings.


IndexError: single positional indexer is out-of-bounds

In [ ]:
embeddings_query_jose = embeddings_query[embeddings_query['id'] == jose['id']]
jose_promedio = np.mean([embeddings_query_jose.iloc[0][:-1].to_numpy(dtype=np.float32), historial_jose], axis=0)
jose_promedio

array([ 2.07595062e-02,  1.69602394e-01,  4.33057249e-02, -1.23935968e-01,
        1.57278582e-01, -8.35606083e-02,  7.35978782e-02,  1.16239160e-01,
        4.73516881e-02,  5.12003480e-03, -1.24026053e-02, -8.79660025e-02,
        4.77575883e-02,  1.08049653e-01, -1.57699585e-01, -1.32207215e-01,
        2.14662567e-01, -3.54450271e-02,  3.53505686e-02,  4.55073640e-02,
       -3.00550200e-02,  2.17229594e-04,  1.84994310e-01, -4.74146195e-02,
        1.75365746e-01, -1.02013245e-01, -1.08271256e-01,  1.25407711e-01,
       -9.81374923e-03, -9.54016112e-03,  1.42388195e-01, -1.87269244e-02,
       -1.02935284e-01,  1.25134036e-01, -2.22097412e-02, -1.27624767e-02,
        1.95628345e-01, -3.28973651e-01, -3.16463597e-02, -1.88383043e-01,
       -1.22352399e-01, -3.52056623e-02,  4.11990210e-02,  5.05440235e-02,
        7.77541101e-02,  5.24775535e-02, -4.94123474e-02,  1.92725006e-03,
       -1.02063604e-01,  1.85610324e-01,  8.43111277e-02, -1.09585784e-02,
       -1.58922702e-01,  

In [ ]:
similares = w2v_model.wv.most_similar(positive=[jose_promedio],topn=10)
similares

[('varias', 0.9908871054649353),
 ('estilo', 0.9905944466590881),
 ('infancia', 0.9897976517677307),
 ('salir', 0.9887130260467529),
 ('bella', 0.9886916279792786),
 ('corazón', 0.9886060953140259),
 ('extraña', 0.9885649085044861),
 ('ir', 0.9885553121566772),
 ('seis', 0.9882989525794983),
 ('habitación', 0.9882380366325378)]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Extract movie embeddings from pelis_embd (assuming the last two columns are 'name' and 'id')
movie_embeddings = pelis_embd.iloc[:, :-2].to_numpy(dtype=np.float32)

# Calculate cosine similarity between jose_promedio and all movie embeddings
# Reshape jose_promedio to be 2D for cosine_similarity function
similarities = cosine_similarity(jose_promedio.reshape(1, -1), movie_embeddings)

# Get the indices of the top 10 most similar movies (descending order)
top_10_indices = similarities.argsort()[0][-10:][::-1]

# Get the corresponding movie names and their similarity scores
similar_movies_df = pd.DataFrame({
    'movie_name': pelis_embd.loc[top_10_indices, 'name'].values,
    'similarity_score': similarities[0, top_10_indices]
})

print("Top 10 Most Similar Movies for Jose:")
display(similar_movies_df)

Top 10 Most Similar Movies for Jose:


,movie_name,similarity_score
0,Más fuerte que su destino,0.998254
1,Todas contra él,0.998119
2,La angustia del miedo,0.998010
3,Un ángel en mi mesa,0.997948
4,El efecto mariposa,0.997947
5,Otoño en Nueva York,0.997826
6,Exorcismo en Connecticut,0.997742
7,Antes de amanecer,0.997740
8,Cuando cae la noche,0.997717
9,Conociendo a Matsuko,0.997674


## Opcion 1:
 Promedio del query con promedio de pelicula del historial contra promedio de pelicula.

## Opcion 2:
  Ponderar Promedio de query junto con el historial de pelicula, y compararlo con el promedio de pelicula.

## Opcion 3:
  Ponerle un peso al historial de peliculas por orden de visualizacion y compararlo con el promedio de pelicula.